The notebook was used to create the blueprint and associate test data for the Fortran-based evaluation engine.  This notebook has no other purpose and can be deleted before publishing this code.

RJB (28 October 2022)

In [ ]:
import os
import random
import time

import numpy as np
import pandas as pd
import pprint

import silence_tensorflow.auto
import tensorflow as tf

import build_model
import experiment_settings
from build_data import build_data
from save_model_run import save_model_run
from save_transfer_blueprint import save_transfer_blueprint
from training_instrumentation import TrainingInstrumentation

In [29]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "28 October 2022"

exp_name = "bivariate_normal_101_EPCP24"
DATA_PATH = "../data/"
MODEL_PATH = "saved_models/"
OVERWRITE_MODEL = True

In [3]:
exp_name = "bivariate_normal_000_EPCP24"

settings = {
    "filename": "nnfit_vlist_02-Jun-2022.dat",
    "uncertainty_type": "bivariate_normal",
    "leadtime": 24,
    "basin": "EP|CP",
    "undersample": False,
    "hiddens": [5, 5],
    "dropout_rate": [0.0, 0.0, 0.0],
    "ridge_param": [0.0, 0.0],
    "learning_rate": 0.001,
    "momentum": 0.9,
    "nesterov": True,
    "batch_size": 64,
    "rng_seed_list": [123],
    "rng_seed": None,
    "act_fun": "relu",
    "n_epochs": 25_000,
    "patience": 50,
    "test_condition": "years",
    "years_test": 2021,
    "val_condition": "random",
    "n_val": 200,
    "n_train": "max",
    "x_names": None,
}

testing_years = 2021
settings["years_test"] = (testing_years,)

rng_seed = 123
settings["rng_seed"] = rng_seed
network_seed = rng_seed

In [4]:
# --------------------- RUN THE EXPERIMENT ---------------------------
# Build the track data tensors for a bivariate normal model.
(
    data_summary,
    x_train,
    onehot_train,
    x_val,
    onehot_val,
    x_test,
    onehot_test,
    x_valtest,
    onehot_valtest,
    df_train,
    df_val,
    df_test,
    df_valtest,
) = build_data(DATA_PATH, settings, verbose=0)

# Define the callbacks
earlystoping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=settings["patience"],
    restore_best_weights=True,
    verbose=1,
)

callbacks = [
    earlystoping_callback,
]

# set random seeds
np.random.seed(rng_seed)
random.seed(rng_seed)
tf.random.set_seed(network_seed)

# Create the model name.
model_name = (
    exp_name
    + "_"
    + str(testing_years)
    + "_"
    + settings["uncertainty_type"]
    + "_"
    + f"network_seed_{network_seed}_rng_seed_{settings['rng_seed']}"
)

# Make, compile, and train the model
tf.keras.backend.clear_session()
model = build_model.make_model(
    settings,
    x_train,
    onehot_train,
    model_compile=True,
)
# model.summary()

# check if the model exists
model_savename = MODEL_PATH + model_name + "_weights.h5"

# train the network
pprint.pprint(model_name)
start_time = time.time()
history = model.fit(
    x_train,
    onehot_train,
    validation_data=(x_val, onehot_val),
    batch_size=settings["batch_size"],
    epochs=settings["n_epochs"],
    shuffle=True,
    verbose=0,
    callbacks=callbacks,
)
stop_time = time.time()

# Display the results, and save the model rum.
best_epoch = np.argmin(history.history["val_loss"])
fit_summary = {
    "network_seed": network_seed,
    "elapsed_time": stop_time - start_time,
    "best_epoch": best_epoch,
    "loss_train": history.history["loss"][best_epoch],
    "loss_valid": history.history["val_loss"][best_epoch],
}

'bivariate_normal_000_EPCP24_2021_bivariate_normal_network_seed_123_rng_seed_123'
Restoring model weights from the end of the best epoch: 110.
Epoch 00160: early stopping


In [5]:
save_model_run(
    data_summary,
    fit_summary,
    model,
    MODEL_PATH,
    model_name,
    settings,
    __version__,
)

save_transfer_blueprint(
    data_summary,
    fit_summary,
    model,
    MODEL_PATH,
    model_name,
    settings,
    __version__,
)

In [6]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 22)]         0           []                               
                                                                                                  
 normalization (Normalization)  (None, 22)           45          ['input_1[0][0]']                
                                                                                                  
 dense (Dense)                  (None, 5)            115         ['normalization[0][0]']          
                                                                                                  
 dense_1 (Dense)                (None, 5)            30          ['dense[0][0]']                  
                                                                                              

In [27]:
print(x_test[0:2])

[[   4.00   50.00   74.70  -10.70  -32.00  -32.00   -8.30  -30.60   47.20
    -8.30  111.40   16.30   63.20   10.00   16.00   15.70   29.10  654.20
     9.80    3.80   -2.20  -11.20]
 [   4.00  125.00   13.40  -29.50   -8.00   24.10   16.70  -16.70  -16.70
    16.70  127.60   15.40  102.00   10.00   14.70    6.90   26.70 1815.20
     0.00   -8.00    4.00    4.00]]


In [28]:
print(x_test[0:2])
model.predict(x_test[0:2])


array([[ 1.88e+01, -5.83e+00,  3.80e+01,  3.81e+01, -2.13e-02],
       [ 2.90e+00, -1.34e+01,  3.26e+01,  3.02e+01,  2.98e-02]],
      dtype=float32)